# Standard pH Calibration (Glass Electrode, Nernstian)

Calibrate a glass pH electrode from buffer measurements and convert your sample
EMF readings into pH values.

The response is assumed **linear** (Nernstian) over the calibrated range:

$$\text{EMF} = E_0 - S\cdot \text{pH}$$

where $S$ is the slope (mV/pH) and $E_0$ the standard potential (mV).

This notebook has **two modes**:

- **Mode A - Build calibration**: fit $E_0$ and $S$ from buffer pH / EMF pairs.
- **Mode B - Predict sample pH**: convert measured sample EMF into pH using the fit.

All input is via **inline arrays** - just edit the values in the marked cells.

*From Saleesongsom et al., ACS Omega (2026).*


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

np.set_printoptions(suppress=True)


## Mode A - Build the calibration

Enter your buffer measurements below. You need **at least 2** points (3+ recommended).
`T_C` is optional (used only to report the theoretical Nernst slope for comparison).


In [ ]:
# ============ EDIT YOUR BUFFER DATA HERE ============
buffer_pH  = [4.009, 7.012, 10.015]     # nominal buffer pH values
buffer_EMF = [181.5, 5.9, -170.9]       # measured EMF (mV)
T_C        = 24.4                        # measurement temperature (deg C), optional
# ====================================================

buffer_pH  = np.asarray(buffer_pH,  float)
buffer_EMF = np.asarray(buffer_EMF, float)

# Linear (Nernstian) regression: EMF = intercept + slope*pH
slope, intercept, r, p, se = stats.linregress(buffer_pH, buffer_EMF)
r2 = r**2
S  = -slope          # reported as a positive slope magnitude (mV/pH)
E0 = intercept       # standard potential (mV)

# Theoretical Nernst slope at this temperature: 2.303*R*T/F
S_theory = 2.302585 * 8.314462 * (T_C + 273.15) / 96485.0 * 1000.0  # mV/pH

print(f"Calibration fit:  EMF = {E0:.2f} - {S:.3f} * pH")
print(f"  slope    S  = {S:.3f} mV/pH")
print(f"  E0          = {E0:.2f} mV")
print(f"  R^2         = {r2:.5f}")
print(f"  theoretical Nernst slope at {T_C} C = {S_theory:.3f} mV/pH "
      f"({100*S/S_theory:.1f}% of Nernstian)")

# store fit for Mode B
calib = dict(S=S, E0=E0, slope=slope, intercept=intercept, r2=r2)


In [ ]:
# ---- Plot the calibration ----
fig, ax = plt.subplots(figsize=(4, 3.2), dpi=120)

x_line = np.array([buffer_pH.min() - 1, buffer_pH.max() + 1])
ax.plot(x_line, intercept + slope * x_line, "-", color="#E8261C", lw=1.6,
        zorder=1, label=f"EMF = {E0:.1f} - {S:.2f}·pH\n$R^2$ = {r2:.4f}")
ax.scatter(buffer_pH, buffer_EMF, s=70, c="#F5A623",
           edgecolors="black", linewidths=1.0, zorder=3)

ax.set_xlabel(r"pH")
ax.set_ylabel("EMF (mV)")
ax.legend(fontsize=8, frameon=False)
ax.set_title("Standard pH calibration")
plt.tight_layout()
plt.show()


## Mode B - Predict sample pH

Enter the EMF you measured for your unknown sample(s). Uses the fit from Mode A:

$$\text{pH} = \dfrac{E_0 - \text{EMF}}{S}$$


In [ ]:
# ============ EDIT YOUR SAMPLE EMF HERE ============
sample_EMF = [120.0, 30.0, -60.0]      # measured EMF of your sample(s), mV
# ===================================================

sample_EMF = np.asarray(sample_EMF, float)
sample_pH  = (calib["E0"] - sample_EMF) / calib["S"]

print("Sample EMF (mV) -> predicted pH")
for e, ph in zip(sample_EMF, sample_pH):
    print(f"  {e:8.2f}  ->  pH {ph:6.3f}")

# Optional: warn if a sample is outside the calibrated pH range
lo, hi = buffer_pH.min(), buffer_pH.max()
out = (sample_pH < lo) | (sample_pH > hi)
if out.any():
    print(f"\nNote: {out.sum()} sample(s) fall outside the calibrated range "
          f"[{lo:.2f}, {hi:.2f}] - extrapolating.")
